## Purpose
As well as *Notebooks* and standard *scripts*, Databricks has a dedicated **SQL Editor**. This is a lightweight, SQL-only workspace designed for writing, running, saving and sharing SQL queries against data in Unity Catalog.

By the end of this section you should be able to:
- Explain what the SQL Editor is and how it differs from Notebooks and scripts
- Open the SQL Editor and attach a SQL warehouse
- Browse data, write and run queries, and use parameters
- Save, share, visualise and schedule a query
- Decide when the SQL Editor is the right tool for the job

## What is the SQL Editor?

The SQL Editor is part of **Databricks SQL**, the area of Databricks aimed at analysts who mainly work in SQL. It gives you a single place to:

- Write SQL with syntax highlighting, autocomplete and formatting
- Browse catalogs, schemas, tables and columns alongside your code
- Run queries on a **SQL warehouse** and view the results in a table
- Turn results into quick visualisations
- Save queries as reusable workspace objects that can be shared, scheduled, and used to power **dashboards** and **alerts**

Each saved item in the SQL Editor is called a **query**. A query lives in your workspace just like a notebook or file (it appears with a `.dbquery` style icon), and can hold one or more SQL statements.

#### SQL warehouses vs clusters
Notebooks and scripts run on a *compute cluster* (or serverless compute). The SQL Editor runs on a **SQL warehouse**, which is compute optimised specifically for SQL workloads. In most cases you will use a shared **serverless SQL warehouse**, which starts in seconds and shuts down automatically when idle, so you are not paying for idle compute.

Because a SQL warehouse only understands SQL, you **cannot** run Python, R or Scala in the SQL Editor, and notebook "magic commands" such as `%python` or `%md` will not work.

## Why would I use the SQL Editor?

1. **Speed for ad-hoc questions** - open a tab, write a `SELECT`, press run. There is no notebook structure to manage.
2. **Data discovery** - the schema browser sits next to your code so you can explore what tables and columns exist and drag names straight into your query.
3. **Reusable, shareable queries** - saved queries can be shared with colleagues with view, run or edit permissions.
4. **Foundation for reporting** - saved queries can feed AI/BI dashboards, SQL alerts and scheduled refreshes.
5. **Low cost, fast start** - serverless SQL warehouses start quickly and are designed for many short SQL queries.

### Notebook vs script vs SQL Editor

| | Notebook | Script (file) | SQL Editor |
|---|---|---|---|
| Languages | Python, SQL, R, Scala, Markdown in one file | One language per file | SQL only |
| Compute | Cluster / serverless | Cluster / serverless | SQL warehouse |
| Best for | Documented, step-by-step analysis and mixed-language work | Modular code, packages, RAP pipelines | Ad-hoc exploration, quick checks, queries behind dashboards and alerts |
| Built-in visualisations | Yes | No | Yes |
| Parameters | Widgets / `:param` | Code arguments | `:param` |

### When not to use the SQL Editor
- When your analysis needs Python or R (e.g. modelling, complex transformations or plotting packages)
- For production pipelines that should follow RAP principles - these belong in version-controlled scripts or notebooks in Git
- When you need lots of narrative documentation alongside your code - a notebook with Markdown cells is better suited

## Opening the SQL Editor

There are a couple ways to get to the SQL Editor:

1. In the left sidebar, under the **SQL** heading, click **SQL Editor** then click **+** tab at the top of the and choose **Query**.
2. In the **Workspace** browser, click **Create** > **Query**. This creates the query directly in that folder.

A new tab called **New Query <timestamp>** will open. Rename it straight away by clicking on the title so it is easy to find later.

#### Attaching a SQL warehouse
Before running anything, check the compute dropdown at the top right of the editor. Select a **SQL warehouse** (for example a serverless starter warehouse). If the warehouse is stopped it will start automatically when you run your first query.

![SQL_editor_compute](../../../images/DatabricksWorkshop/SQL_editor_server.png)


#### Setting the default catalog and schema
Next to the warehouse selector there are dropdowns for the default **catalog** and **schema**. If you set these you can write `SELECT * FROM games_ranking` instead of the fully qualified `catalog.schema.table` name. For shared or saved queries it is best practice to always use the full three-level name so the query works for everyone.

![SQL_editor_catalog_dropdown](../../../images/DatabricksWorkshop/SQL_editor_catalog_dropdown.png)


## A tour of the interface

| Area | What it does |
|---|---|
| **Schema browser** (left panel) | Browse catalogs, schemas, tables and columns. Hover over a table and click `>>` to drop its name into your query. |
| **Query tabs** (top) | Each open query is a tab, similar to a browser. Unsaved changes are kept between sessions. |
| **Editor pane** | Where you write SQL. Supports autocomplete (start typing a table or column name), comments (`--` or `/* */`) and formatting. |
| **Run button** | Runs the whole query, or only the highlighted text. The dropdown next to it controls the row **limit** (1,000 by default). |
| **Results pane** (bottom) | Shows the output as a table. You can sort, filter, search, copy and download results, and add visualisations with the **+** next to the Results tab. |
| **Genie Code / Assistant** | AI help to write, explain or fix SQL. |
| **Save / Share / Schedule** (top right) | Save the query, share it with others, or schedule it to refresh. |

#### Useful keyboard shortcuts
- **Ctrl + Enter** - run the query, or the highlighted selection
- **Ctrl + Shift + Enter** - run only the statement the cursor is in
- **Ctrl + S** - save
- **Ctrl + /** - comment / uncomment the selected lines
- **Ctrl + Shift + F** - format the query

#### Running multiple statements
A single query tab can contain several statements separated by `;`. When you run the whole tab, every statement runs in order and each result appears in its own results tab. Highlight just one statement to run only that one - this is a very common way of working when exploring data.

## Writing your first queries

The queries below use the training data in `catalog_40_copper_analyst_training.steam`. They will run in this notebook, but the aim is for you to **copy them into a new query in the SQL Editor** and run them there so you can compare the experience.

#### 1. Look at the data
Always start by previewing a table. Using `LIMIT` keeps it fast.

In [0]:
-- Preview the first 10 rows of the games ranking table
SELECT *
FROM catalog_40_copper_analyst_training.steam.games_ranking
LIMIT 10;

#### 2. Filter and sort
Find the top 10 Strategy games by revenue.

In [0]:
-- Top 10 Strategy games ranked by revenue
SELECT game_name, genre, rank
FROM catalog_40_copper_analyst_training.steam.games_ranking
WHERE genre = 'Strategy'
  AND rank_type = 'Revenue'
ORDER BY rank
LIMIT 10;

#### 3. Aggregate
Summarise the reviews table to see which games have the most reviews, how long reviewers played on average, and what percentage recommend the game. This is a good query to try a **visualisation** on in the SQL Editor (for example a bar chart of `pct_recommended` by `game_name`).

In [0]:
-- Summary of reviews per game
SELECT
  game_name,
  COUNT(*)                                                                    AS number_of_reviews,
  ROUND(AVG(hours_played), 1)                                                 AS avg_hours_played,
  ROUND(100 * AVG(CASE WHEN recommendation = 'Recommended' THEN 1 ELSE 0 END), 1) AS pct_recommended
FROM catalog_40_copper_analyst_training.steam.steam_game_reviews
GROUP BY game_name
ORDER BY number_of_reviews DESC, pct_recommended DESC
LIMIT 20;

## Using parameters

Parameters let you (or anyone you share the query with) change a value without editing the SQL. In the SQL Editor you add a parameter by typing a colon followed by a name, e.g. `:genre`. As soon as you do, an input box appears above the results pane.

```sql
SELECT game_name, rank
FROM catalog_40_copper_analyst_training.steam.games_ranking
WHERE genre = :genre
  AND rank_type = :rank_type
ORDER BY rank
LIMIT 10;
```

Click the cog icon on a parameter to change its **type** (text, number, date, dropdown list etc.). A **dropdown list** is useful here - set the values to `Action, Adventure, Role-Playing, Simulation, Sports & Racing, Strategy`.

This is the same `:parameter_name` syntax you saw for widgets in *02 Using Databricks Notebooks*, so queries written this way are easy to move between notebooks and the SQL Editor.

## Saving, sharing and reusing queries

#### Saving
Click **Save** (or Ctrl + S). Choose a folder in your workspace - keep workshop work in your own `workshop_practice` folder. Saved queries can be found again from the **Queries** list in the SQL section of the sidebar, or in the Workspace browser.

#### Visualisations
After running a query, click the **+** next to the Results tab and choose **Visualization**. Pick a chart type (bar, line, pie, map etc.), choose the X and Y columns and save. Visualisations are stored with the query.

#### Sharing
Click **Share** to give other users or groups permission to view, run, edit or manage the query. Remember that sharing a query does not grant access to the underlying data - users still need permissions on the tables in Unity Catalog.

#### Scheduling
Click **Schedule** to refresh a saved query on a regular basis (e.g. daily at 8am). Saved queries can also be added as a SQL task in a **Job**, which is covered in *04 Using Jobs & Pipelines*.

#### Dashboards and alerts
Saved queries and their SQL are the building blocks of **AI/BI dashboards** and **SQL alerts** (e.g. email me if a row count drops to zero).

#### Query History
The **Query History** page (under SQL in the sidebar) shows every query you have run, its status, duration, and the warehouse used. It is useful for finding a query you ran earlier but did not save, and for spotting slow queries.

#### Version control
Unlike notebooks and files, queries are not as naturally suited to Git workflows. If a query becomes part of a regular process, consider moving the SQL into a `.sql` script or notebook in your Git repository (see *01 Using Databricks with Git*).

### ✅ Task: Explore Steam data in the SQL Editor

1. In the left sidebar, go to **SQL** > **SQL Editor** and open a new query. Rename it `03_my_first_query`.
2. Attach a **SQL warehouse** using the compute dropdown at the top right.
3. Use the **schema browser** to find `catalog_40_copper_analyst_training` > `steam` and look at the columns available in each of the three tables.
4. Write a query that previews 10 rows of `games_description`. Use the schema browser to insert the table name rather than typing it.
5. In the same tab, write a second query that returns the **top 5 games by `Sales`** for the `Action` genre from `games_ranking`. Highlight and run only this second statement.
6. Change the hard-coded genre to a parameter called `:genre` and test it with at least two different genres.
7. Copy the *Review summary by game* query from above into a new tab, run it, and add a **bar chart** visualisation of `pct_recommended` by `game_name`.
8. **Save** both queries into your `workshop_practice` folder.
9. Open **Query History** and find the queries you just ran. How long did they take?

**Stretch:** Join `games_ranking` to `steam_game_reviews` on `game_name` to find the average `hours_played` for each `genre`. Which genre's players play the longest?

**Questions to consider:**
- What happened when you tried to run a statement starting with `%python` in the SQL Editor? Why?
- What changed when you set a default catalog and schema? Why should you still use full three-level names in shared queries?
- Which tasks felt quicker in the SQL Editor than in a notebook, and which would you rather do in a notebook or script?

In [0]:
%py
with open("Quiz Using the SQL Editor.html", "r", encoding="utf-8") as f:
    html_content = f.read()
displayHTML(html_content)